In [1]:
# ═══════════════════════════════════════════════════════════════════
# CELLULE 1 — Installation (exécute en premier !)
# ═══════════════════════════════════════════════════════════════════
import subprocess
subprocess.run(['pip', 'install', 'lightgbm', '-q'], check=True)
print("✅ LightGBM installé")

# ═══════════════════════════════════════════════════════════════════
# CELLULE 2 — Code principal
# ═══════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import lightgbm as lgb
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import glob, os, warnings, time
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")

# ── 1. Load Data ─────────────────────────────────────────────────
def find_file(filename):
    matches = glob.glob(f'/kaggle/input/**/{filename}', recursive=True)
    if matches:
        return matches[0]
    raise FileNotFoundError(filename)

DATA_DIR = os.path.dirname(find_file('train.csv'))
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sub   = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
print(f"Train: {train.shape} | Test: {test.shape}")

# ── 2. Feature Engineering ────────────────────────────────────────
def make_features(df, loc_stats=None, fit=False):
    df = df.copy()

    # Cyclical time
    df['hr_sin']   = np.sin(2*np.pi*df['HR']/24)
    df['hr_cos']   = np.cos(2*np.pi*df['HR']/24)
    df['mo_sin']   = np.sin(2*np.pi*df['MO']/12)
    df['mo_cos']   = np.cos(2*np.pi*df['MO']/12)
    df['dy_sin']   = np.sin(2*np.pi*df['DY']/31)
    df['dy_cos']   = np.cos(2*np.pi*df['DY']/31)
    df['doy']      = (df['MO']-1)*30.4375 + df['DY']
    df['doy_sin']  = np.sin(2*np.pi*df['doy']/365)
    df['doy_cos']  = np.cos(2*np.pi*df['doy']/365)

    # Time flags
    df['is_day']      = ((df['HR']>=6) & (df['HR']<=18)).astype(float)
    df['is_night']    = 1.0 - df['is_day']
    df['is_morning']  = ((df['HR']>=5) & (df['HR']<=10)).astype(float)
    df['is_evening']  = ((df['HR']>=16) & (df['HR']<=22)).astype(float)
    df['is_midnight'] = ((df['HR']>=23) | (df['HR']<=4)).astype(float)

    # Season
    season_map = {1:1,2:1,3:2,4:2,5:2,6:3,7:3,8:3,9:4,10:4,11:4,12:1}
    df['season']     = df['MO'].map(season_map).astype(float)
    df['season_sin'] = np.sin(2*np.pi*df['season']/4)
    df['season_cos'] = np.cos(2*np.pi*df['season']/4)

    # Rainy season flags
    df['is_oct']   = (df['MO']==10).astype(float)
    df['is_nov']   = (df['MO']==11).astype(float)
    df['is_dec']   = (df['MO']==12).astype(float)
    df['is_rainy_season']   = df['MO'].isin([10,11,12,1,2]).astype(float)
    df['is_peak_rain']      = df['MO'].isin([10,11,12]).astype(float)
    df['rain_season_weight']= df['MO'].map(
        {1:0.6,2:0.5,3:0.2,4:0.1,5:0.05,6:0.0,
         7:0.0,8:0.0,9:0.1,10:0.8,11:0.9,12:0.7}
    )

    # Spatial features
    df['loc_id']    = df.groupby(['LON','LAT']).ngroup()
    df['lon_norm']  = (df['LON']-(-5.83))/(8.76-(-5.83))
    df['lat_norm']  = (df['LAT']-23.01)/(36.82-23.01)
    df['north_idx'] = df['lat_norm']
    df['lon_x_lat'] = df['LON']*df['LAT']

    # Climate zones
    df['zone_north']  = (df['LAT'] > 34).astype(float)
    df['zone_middle'] = ((df['LAT'] > 30) & (df['LAT'] <= 34)).astype(float)
    df['zone_sahara'] = (df['LAT'] <= 30).astype(float)

    # Spatial × temporal interactions
    df['north_x_rain'] = df['zone_north'] * df['is_rainy_season']
    df['north_x_peak'] = df['zone_north'] * df['is_peak_rain']
    df['sahara_x_rain']= df['zone_sahara'] * df['is_rainy_season']

    # Per-location statistics
    if fit:
        stats = df.groupby('loc_id').agg(
            rh_mean=('RH2M','mean'), rh_std=('RH2M','std'),
            ws_mean=('WS2M','mean'), ws_std=('WS2M','std'),
        ).reset_index()
        loc_stats = stats.set_index('loc_id').to_dict('index')

    if loc_stats:
        df['loc_rh_mean'] = df['loc_id'].map(lambda l: loc_stats.get(l,{}).get('rh_mean',50))
        df['loc_rh_std']  = df['loc_id'].map(lambda l: loc_stats.get(l,{}).get('rh_std',10))
        df['loc_ws_mean'] = df['loc_id'].map(lambda l: loc_stats.get(l,{}).get('ws_mean',3))
        df['loc_ws_std']  = df['loc_id'].map(lambda l: loc_stats.get(l,{}).get('ws_std',1))
        df['rh2m_anom']   = df['RH2M'] - df['loc_rh_mean']
        df['ws2m_anom']   = df['WS2M'] - df['loc_ws_mean']

    # Meteorological interactions
    df['rh_x_ws']     = df['RH2M'] * df['WS2M']
    df['rh_sq']       = df['RH2M'] ** 2
    df['ws_sq']       = df['WS2M'] ** 2
    df['rh_x_day']    = df['RH2M'] * df['is_day']
    df['rh_x_night']  = df['RH2M'] * df['is_night']
    df['rh_x_north']  = df['RH2M'] * df['zone_north']
    df['high_rh']     = (df['RH2M'] > 70).astype(float)
    df['very_high_rh']= (df['RH2M'] > 85).astype(float)
    df['extreme_rh']  = (df['RH2M'] > 92).astype(float)
    df['storm_idx']   = df['WS2M'] * (df['RH2M']/100) ** 2
    df['rain_signal'] = df['zone_north'] * df['very_high_rh'] * df['is_rainy_season']
    df['rain_signal2']= df['north_idx']  * (df['RH2M']/100)  * df['rain_season_weight']

    return df, loc_stats

loc_stats = None
train, loc_stats = make_features(train, fit=True)
test,  _         = make_features(test, loc_stats=loc_stats, fit=False)

FCOLS = [
    'LON','LAT','loc_id','lon_norm','lat_norm','north_idx','lon_x_lat',
    'zone_north','zone_middle','zone_sahara',
    'HR','MO','DY',
    'hr_sin','hr_cos','mo_sin','mo_cos','dy_sin','dy_cos',
    'doy','doy_sin','doy_cos',
    'is_day','is_night','is_morning','is_evening','is_midnight',
    'season','season_sin','season_cos',
    'is_oct','is_nov','is_dec',
    'is_rainy_season','is_peak_rain','rain_season_weight',
    'north_x_rain','north_x_peak','sahara_x_rain',
    'RH2M','WS2M',
    'loc_rh_mean','loc_rh_std','loc_ws_mean','loc_ws_std',
    'rh2m_anom','ws2m_anom',
    'rh_x_ws','rh_sq','ws_sq','rh_x_day','rh_x_night','rh_x_north',
    'high_rh','very_high_rh','extreme_rh','storm_idx',
    'rain_signal','rain_signal2',
]

X_tr = train[FCOLS]
X_te = test[FCOLS]
y_temp    = train['temperature'].values
y_prec    = train['precipitation'].values
y_prec_log= np.log1p(y_prec)
y_rain    = (y_prec > 0.01).astype(int)
print(f"Features: {len(FCOLS)} | Rain rate: {y_rain.mean():.2%}")

# ── 3. LightGBM ──────────────────────────────────────────────────
BASE_P = dict(
    n_estimators=1200, learning_rate=0.04,
    num_leaves=255, max_depth=-1,
    min_child_samples=15,
    feature_fraction=0.85, bagging_fraction=0.85, bagging_freq=5,
    reg_alpha=0.05, reg_lambda=0.1,
    n_jobs=-1, verbose=-1, random_state=SEED,
)

print("\n[LightGBM] Temperature ...")
t0 = time.time()
lgb_temp = lgb.LGBMRegressor(**BASE_P, objective='mae')
lgb_temp.fit(X_tr, y_temp, eval_set=[(X_tr, y_temp)],
             callbacks=[lgb.log_evaluation(200)])
print(f"  Done in {time.time()-t0:.1f}s")

print("\n[LightGBM] Rain Classifier ...")
t0 = time.time()
lgb_rain_cls = lgb.LGBMClassifier(**BASE_P, objective='binary', metric='binary_logloss')
lgb_rain_cls.fit(X_tr, y_rain, eval_set=[(X_tr, y_rain)],
                 callbacks=[lgb.log_evaluation(200)])
print(f"  Done in {time.time()-t0:.1f}s")

print("\n[LightGBM] Rain Amount (rainy rows only) ...")
t0 = time.time()
rain_mask = y_rain == 1
lgb_rain_amt = lgb.LGBMRegressor(**BASE_P, objective='mae')
lgb_rain_amt.fit(X_tr[rain_mask], y_prec_log[rain_mask],
                 eval_set=[(X_tr[rain_mask], y_prec_log[rain_mask])],
                 callbacks=[lgb.log_evaluation(200)])
print(f"  Done in {time.time()-t0:.1f}s")

def predict_precip(X, threshold=0.20):
    rain_prob   = lgb_rain_cls.predict_proba(X)[:, 1]
    rain_amount = np.expm1(lgb_rain_amt.predict(X))
    rain_amount = np.clip(rain_amount, 0, None)
    return np.where(rain_prob > threshold, rain_prob * rain_amount, 0.0)

lgb_pred_temp = lgb_temp.predict(X_te)
lgb_pred_prec = predict_precip(X_te)

mae_t = np.mean(np.abs(lgb_temp.predict(X_tr) - y_temp))
mae_p = np.mean(np.abs(predict_precip(X_tr) - y_prec))
print(f"\n[LightGBM] MAE temp={mae_t:.4f} prec={mae_p:.4f} WMCAE={0.25*mae_t+0.75*mae_p:.4f}")

# ── 4. GRU Model ─────────────────────────────────────────────────
scaler_X    = StandardScaler()
X_tr_sc     = scaler_X.fit_transform(X_tr.values.astype(np.float32))
X_te_sc     = scaler_X.transform(X_te.values.astype(np.float32))
scaler_temp = StandardScaler()
scaler_prec = StandardScaler()
y_temp_sc   = scaler_temp.fit_transform(y_temp.reshape(-1,1)).flatten().astype(np.float32)
y_prec_sc   = scaler_prec.fit_transform(y_prec_log.reshape(-1,1)).flatten().astype(np.float32)
loc_ids_tr  = train['loc_id'].values
loc_ids_te  = test['loc_id'].values

class WDS(Dataset):
    def __init__(self, X, yt, yp, lid):
        self.X   = torch.FloatTensor(X)
        self.yt  = torch.FloatTensor(yt)
        self.yp  = torch.FloatTensor(yp)
        self.lid = torch.LongTensor(lid)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.yt[i], self.yp[i], self.lid[i]

class WTS(Dataset):
    def __init__(self, X, lid):
        self.X   = torch.FloatTensor(X)
        self.lid = torch.LongTensor(lid)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.lid[i]

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim*2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim*2, dim), nn.LayerNorm(dim),
        )
        self.act = nn.GELU()
    def forward(self, x):
        return self.act(x + self.net(x))

class SpatiotemporalGRU(nn.Module):
    def __init__(self, n_feat, n_locs=48, emb_dim=32,
                 hidden=512, gru_h=128, n_blocks=6, dropout=0.1):
        super().__init__()
        self.loc_emb = nn.Embedding(n_locs, emb_dim)
        in_dim       = n_feat + emb_dim
        self.proj    = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), nn.GELU())
        self.gru     = nn.GRU(hidden, gru_h, num_layers=2,
                              batch_first=True, bidirectional=True, dropout=dropout)
        self.merge   = nn.Sequential(
            nn.Linear(hidden+gru_h*2, hidden), nn.LayerNorm(hidden), nn.GELU())
        self.trunk   = nn.Sequential(*[ResBlock(hidden, dropout) for _ in range(n_blocks)])
        self.temp_head = nn.Sequential(
            nn.Linear(hidden,256), nn.GELU(), nn.Dropout(0.05), nn.Linear(256,1))
        self.prec_head = nn.Sequential(
            nn.Linear(hidden,256), nn.GELU(), nn.Dropout(0.05), nn.Linear(256,1))

    def forward(self, x, lid):
        emb  = self.loc_emb(lid)
        x    = torch.cat([x, emb], dim=-1)
        h    = self.proj(x)
        g, _ = self.gru(h.unsqueeze(1))
        g    = g.squeeze(1)
        h    = self.merge(torch.cat([h, g], dim=-1))
        h    = self.trunk(h)
        return self.temp_head(h).squeeze(-1), self.prec_head(h).squeeze(-1)

class WMAE(nn.Module):
    def forward(self, pt, pp, yt, yp):
        return 0.25*(pt-yt).abs().mean() + 0.75*(pp-yp).abs().mean()

BATCH  = 8192
tr_ds  = WDS(X_tr_sc, y_temp_sc, y_prec_sc, loc_ids_tr)
tr_dl  = DataLoader(tr_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
te_ds  = WTS(X_te_sc, loc_ids_te)
te_dl  = DataLoader(te_ds, batch_size=BATCH, shuffle=False, num_workers=2)

EPOCHS = 50
model  = SpatiotemporalGRU(X_tr_sc.shape[1], n_locs=48, emb_dim=32,
                            hidden=512, gru_h=128, n_blocks=6, dropout=0.1).to(DEVICE)
print(f"\n[GRU] Params: {sum(p.numel() for p in model.parameters()):,}")

criterion = WMAE()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-3, steps_per_epoch=len(tr_dl), epochs=EPOCHS)

best_loss, best_state = float('inf'), None
print(f"[GRU] Training {EPOCHS} epochs ...")

for ep in range(1, EPOCHS+1):
    model.train()
    ep_loss = 0
    for xb, ytb, ypb, lid in tr_dl:
        xb  = xb.to(DEVICE)
        ytb = ytb.to(DEVICE)
        ypb = ypb.to(DEVICE)
        lid = lid.to(DEVICE)
        pt, pp = model(xb, lid)
        loss   = criterion(pt, pp, ytb, ypb)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        ep_loss += loss.item()

    avg = ep_loss / len(tr_dl)
    if avg < best_loss:
        best_loss  = avg
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if ep % 10 == 0:
        print(f"  Epoch {ep:3d}/{EPOCHS}  loss={avg:.5f}  lr={scheduler.get_last_lr()[0]:.2e}")

model.load_state_dict(best_state)
model.eval()
dl_t, dl_p = [], []
with torch.no_grad():
    for xb, lid in te_dl:
        pt, pp = model(xb.to(DEVICE), lid.to(DEVICE))
        dl_t.append(pt.cpu().numpy())
        dl_p.append(pp.cpu().numpy())

dl_pred_temp = scaler_temp.inverse_transform(
    np.concatenate(dl_t).reshape(-1,1)).flatten()
dl_pred_prec = np.expm1(scaler_prec.inverse_transform(
    np.concatenate(dl_p).reshape(-1,1)).flatten())
dl_pred_prec = np.clip(dl_pred_prec, 0, None)

# ── 5. Ensemble ───────────────────────────────────────────────────
W_LGB_T = 0.40; W_GRU_T = 0.60
W_LGB_P = 0.60; W_GRU_P = 0.40

ens_temp = W_LGB_T * lgb_pred_temp + W_GRU_T * dl_pred_temp
ens_prec = W_LGB_P * lgb_pred_prec + W_GRU_P * dl_pred_prec
ens_prec = np.clip(ens_prec, 0, None)

print(f"\n[Ensemble] Temp  min={ens_temp.min():.1f} max={ens_temp.max():.1f} mean={ens_temp.mean():.1f}")
print(f"[Ensemble] Prec  min={ens_prec.min():.4f} max={ens_prec.max():.2f} mean={ens_prec.mean():.4f}")
print(f"[Ensemble] %zero-prec: {(ens_prec < 0.01).mean():.2%}")

# ── 6. Submission ─────────────────────────────────────────────────
test['temperature']   = ens_temp
test['precipitation'] = ens_prec

final_sub = sub[['id']].merge(
    test[['id','temperature','precipitation']], on='id', how='left')

assert final_sub.isnull().sum().sum() == 0, "NaN in submission!"
assert (final_sub['precipitation'] >= 0).all(), "Negative precipitation!"

final_sub.to_csv('/kaggle/working/submission.csv', index=False)
print(f"\n✅ Submission saved! {len(final_sub)} rows")
print(final_sub.head(10))


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


✅ LightGBM installé


Device : cpu


Train: (315648, 11) | Test: (105984, 9)


Features: 59 | Rain rate: 25.06%

[LightGBM] Temperature ...


...

  Epoch  30/50  loss=0.09263  lr=1.83e-03


  Epoch  40/50  loss=0.06323  lr=5.62e-04


  Epoch  50/50  loss=0.05296  lr=1.60e-08



[Ensemble] Temp  min=6.4 max=38.2 mean=20.9
[Ensemble] Prec  min=0.0000 max=36.54 mean=0.5237
[Ensemble] %zero-prec: 52.34%



✅ Submission saved! 105984 rows
                                     id  temperature  precipitation
0  179faf24-c337-41eb-bc16-ed374c0f5041    28.192728       0.002585
1  13ad071f-24fc-421b-a0c7-a199595e1949    27.595636       0.002304
2  13f0f081-836e-4d48-8499-e5c9cb072f1e    26.876407       0.002105
3  d6688299-2051-4c2d-8e48-12941e1ca6c9    26.296689       0.001998
4  6c4dedb2-7ce0-4e3b-a29b-882da767904d    25.752563       0.002108
5  0692db8b-9215-47f6-ab96-dbc08745a5d5    25.126391       0.001902
6  b08e5356-e8ba-4a8d-a8af-ce3c3461741b    24.965079       0.002093
7  1e7e9256-1b4f-443c-93fc-d3472f65eb8b    26.448908       0.002133
8  4061dc7a-affb-4dec-b646-8dd7c43a5544    28.912171       0.002363
9  0dbc65af-376b-437a-9e9b-0242124b5812    31.474322       0.001309
